# K-Nearest Neighbors (KNN) Example

This notebook demonstrates the k-Nearest Neighbors classification algorithm using the Iris dataset. It includes dataset exploration, preprocessing, model training, prediction, evaluation, and visualization.


In [ ]:
# Section 1: Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

sns.set(style="whitegrid")

## Load and Explore the Dataset

Load the Iris dataset and inspect the feature names, sample records, and class distribution.


In [ ]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")

print("Dataset shape:", X.shape)
print("Target classes:", iris.target_names)

# Display first rows and basic statistics
X.head(), X.describe(), y.value_counts()

## Split Data into Training and Testing Sets

Use an 80-20 split to create training and testing subsets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

## Normalize the Features

Scale features using `StandardScaler` to ensure distance calculations are balanced across dimensions.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature means after scaling (train):", np.round(X_train_scaled.mean(axis=0), 3))
print("Feature std after scaling (train):", np.round(X_train_scaled.std(axis=0), 3))

## Train the KNN Model

Train a `KNeighborsClassifier` with `k=5` using the normalized training data.


In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

print("Model trained with k=5")

## Make Predictions

Use the trained model to predict labels on the test set and show a few sample predictions.


In [ ]:
y_pred = knn.predict(X_test_scaled)
y_proba = knn.predict_proba(X_test_scaled)

sample_results = pd.DataFrame(
    {
        "true_label": y_test.values,
        "predicted_label": y_pred,
        "confidence": np.max(y_proba, axis=1),
    }
)

sample_results.head(10)

## Evaluate Model Performance

Compute accuracy, precision, recall, F1-score, and display a confusion matrix for the test results.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro")
recall = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## Visualize Results

Visualize the confusion matrix and compare KNN performance across several values of `k`.


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

k_values = list(range(1, 16))
accuracy_scores = []
for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_scaled, y_train)
    y_pred_k = knn_k.predict(X_test_scaled)
    accuracy_scores.append(accuracy_score(y_test, y_pred_k))

plt.figure(figsize=(8, 4))
plt.plot(k_values, accuracy_scores, marker="o")
plt.title("KNN Accuracy for Different k Values")
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Accuracy")
plt.xticks(k_values)
plt.grid(True)
plt.show()